# Step 3 · Policies and the review agent
PayNest's rules live in `config/standards/` (readable version: `../policies/`). A LangGraph workflow checks
the contract against all 10, Python verifies and scores, and the graph **pauses for a lawyer**.

> **Start Jupyter through the project script** so it gets the same settings as the app:
>
> ```bash
> cd copilot
> ./scripts/run.sh jupyter lab
> ```

In [ ]:
import sys, os
sys.path.insert(0, "..")          # make the copilot's `app` package importable
os.chdir("..")                    # run from copilot/, like the app does
print("Models are called at:", os.getenv("BIFROST_URL"), "| profile:", os.getenv("SETTINGS_PROFILE") or "direct to Ollama")

## 1. The company's rules

In [ ]:
from app.core.config import standards
for r in standards():
    print(f"{r['id']:<4} w{r['weight']} {'KEY' if r['critical'] else '   '}  {r['title']}")

## 2. Run a review (about 5–10 minutes on a laptop: one model call per standard)

In [ ]:
from pathlib import Path
from app.ingest.ingest_service import ingest_pdf
from app.graph import policy_service
pdf = Path("sample_contracts/01_carewell_hospitals.pdf")
doc = ingest_pdf(pdf.name, pdf.read_bytes())
review = policy_service.start(doc["doc_id"], reviewer="Student")
review["state"], review["system_result"]["score"], review["system_result"]["band"]

## 3. What the model said, and what Python decided

In [ ]:
for f in review["findings"]:
    print(f"{f['rule_id']:<4} model: {f['llm_status']:<12} final: {f['status']:<13} {f['reason'][:70]}")

## 4. The graph is paused at `interrupt()`. Decide every flagged finding, then resume

In [ ]:
print("Waiting for decisions on:", review["flagged"], "| because:", review["gate_reasons"])
decisions = [{"rule_id": rid, "action": "approve"} for rid in review["flagged"]]
final = policy_service.resume(review["review_id"], decisions)
final["state"], final["report"]["final_result"]["score"], final["report"]["final_result"]["band"]

**Takeaway:** the model only labels; every quote is checked against the PDF, the score is plain arithmetic, and a person signs off on risk. Try the same in the portal: `make up` → http://localhost:8000.